# Measurement DB response preprocessing

Load all competition splits in the Hugging Face `responses` and `subjects` datasets, and create dicts mapping hashes to human-readable identifiers/content.

This dataset is gated. Accept its access terms on Hugging Face and authenticate in the notebook environment first.

In [12]:
import random
import datasets
import utils

from huggingface_hub import notebook_login
notebook_login()

RANDOM_SEED = 5814
random.seed(RANDOM_SEED)

benchmarks = (
    "multi_swebench+real_webagents+"
    "researchcodebench+swe_rebench"
)

SAMPLE_SIZE = 1000

# ---------------------------------------------------------
# Create lookup tables (items + subjects)
# ---------------------------------------------------------
print("Caching items by item_id")
items_ds = datasets.load_dataset(
    "aims-foundations/measurement-db",
    name="items",
    split=benchmarks,
    token=True
)
items_lookup = {row["item_id"]: row for row in items_ds}
print(f"Items cached: {len(items_lookup)}")

print("Caching subjects by subject_id")
subjects_ds = datasets.load_dataset(
    "aims-foundations/measurement-db",
    name="subjects",
    split=benchmarks,
    token=True
)
subjects_lookup = {row["subject_id"]: row for row in subjects_ds}
print(f"Subjects cached: {len(subjects_lookup)}")


Caching items by item_id
Items cached: 8878
Caching subjects by subject_id
Subjects cached: 134


Generate one-hot vector representations for subjects

In [13]:
# dict.fromkeys preserves first-seen order while removing duplicate names.
normalized_names = list(
    dict.fromkeys(
        subject["normalized_name"]
        for subject in subjects_lookup.values()
        if subject.get("normalized_name") is not None
    )
)
num_names = len(normalized_names)
name_to_one_hot = {
    name: [int(index == position) for index in range(num_names)]
    for position, name in enumerate(normalized_names)
}

# Verify each vector's size, one-hot value, and uniqueness.
assert all(
    len(vector) == num_names and sum(vector) == 1
    for vector in name_to_one_hot.values()
)
assert len({tuple(vector) for vector in name_to_one_hot.values()}) == num_names

print(f"Created {num_names} unique name vectors; each has length {num_names}.")

Created 60 unique name vectors; each has length 60.


Sample from the `responses` dataset and merge in data from `subjects` and `items`

In [14]:

# ---------------------------------------------------------
# Streaming responses + reservoir sampling
# ---------------------------------------------------------
print("Streaming responses and performing reservoir sampling...")
responses_stream = datasets.load_dataset(
    "aims-foundations/measurement-db",
    name="responses",
    split=benchmarks,
    streaming=True,
    token=True
)

def _merge_row(example):
    item = items_lookup.get(example["item_id"])
    subject = subjects_lookup.get(example["subject_id"])
    if item is None or subject is None:
        return None
    return {**example, **item, **subject}

joined_stream = responses_stream.map(_merge_row)

reservoir = []
n_seen = 0

for row in joined_stream:
    if row is None:
        continue

    # Validate fields early
    content = row.get("content")
    subject_name = row.get("normalized_name")
    response_val = row.get("response")

    if not isinstance(content, str):
        continue
    if not isinstance(subject_name, str):
        continue
    if response_val is None:
        continue

    # Reservoir sampling
    if len(reservoir) < SAMPLE_SIZE:
        reservoir.append(row)
    else:
        j = random.randint(0, n_seen)
        if j < SAMPLE_SIZE:
            reservoir[j] = row

    n_seen += 1

print(f"Reservoir sampled {len(reservoir)} rows from ~{n_seen} streamed rows")

for item in reservoir[:5]:
    print(item)


Streaming responses and performing reservoir sampling...
Reservoir sampled 1000 rows from ~136282 streamed rows
{'subject_id': '58c3de46be1beb60', 'item_id': '0a017ffe57db9265', 'response_id': '1c80729da617b63a', 'benchmark_id': 'swe_rebench', 'trial': 3, 'test_condition': None, 'interactors': None, 'response': 0.0, 'raw_item_id': 'conda-incubator__conda-recipe-manager-49', 'content': 'Add `merge_build_host` transformation\nWe are missing a transformation of the `merge_build_host` field from CEP-14:\r\n```\r\n  # merge the build and host environments (used in many R packages on Windows)\r\n  # was `merge_build_host`\r\n  merge_build_and_host_envs: bool (defaults to false)\r\n```\r\n\r\n`rattler-build` error:\r\n```\r\nError:   × Parsing: invalid field `merge_build_host`.\r\n    ╭─[50:3]\r\n 49 │   noarch: generic\r\n 50 │   merge_build_host: $${{ true if win }}\r\n    ·   ────────┬───────\r\n    ·           ╰── here\r\n 51 │   rpaths:\r\n    ╰────\r\n```', 'asset_manifest': None, 'grad

Merge one-hot subject vectors to reservoir samples

In [15]:
# Map one-hot subject vectors back to reservoir samples
for sample in reservoir:
        sample["one_hot_normalized_name"] = name_to_one_hot[sample["normalized_name"]]

for sample in reservoir[:5]:
    print(sample["one_hot_normalized_name"])

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


Encode with Approach 1 chunking: 512-token windows, at most 2,048 content tokens, token-weighted normalized pooling. Training and prediction share `models/text_chunking.py`.

In [ ]:
embeddings = utils.embed_text(
    utils.get_sbert(), [sample["content"] for sample in reservoir], show_progress=True
)
for sample, vector in zip(reservoir, embeddings):
    sample["content_embedding"] = vector
    sample["text_encoding"] = utils.TEXT_ENCODING

for sample in reservoir[:5]:
    print(sample["content_embedding"])


Save chunked samples separately from Steven's baseline `samples.json`. For the same saved 1,000 responses without re-sampling, run `python reembed_samples.py` instead of this notebook.

In [ ]:
import json
from pathlib import Path

import numpy as np

class NumpyJSONEncoder(json.JSONEncoder):
    def default(self, value):
        if isinstance(value, np.ndarray):
            return value.tolist()
        if isinstance(value, np.generic):
            return value.item()
        return super().default(value)

output_path = Path("samples_chunked.json")
with output_path.open("x", encoding="utf-8") as file:
    json.dump(reservoir, file, cls=NumpyJSONEncoder, ensure_ascii=False, indent=2)

print(f"Saved {len(reservoir)} samples to {output_path.resolve()}")

Confirm json loads into a Pandas dataframe

In [ ]:
import pandas as pd
df = pd.read_json("samples_chunked.json")
print(df.head())